# 17 — Hitta dolda medium-bilder i epic-mappen
Kör epic-modellen på sin egen `epic`-mapp och sorterar efter `p_medium` fallande. Bilder med högst `p_medium` (även om epic fortfarande vinner) är de bästa kandidaterna för felmärkningar — samma teknik som notebook 16 använde för clean/mustache.

In [6]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image

EPIC_DIR = 'data/epic_dataset/epic'
MODEL_PATH = 'models/epic_detector_3class_2.keras'
IMG_SIZE = (178, 178)

model = tf.keras.models.load_model(MODEL_PATH)

def collect_files(folder):
    paths = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                paths.append(os.path.join(root, f))
    return paths

files = collect_files(EPIC_DIR)
print(f'{len(files)} bilder i epic-mappen.')

1437 bilder i epic-mappen.


## Kör modellen på alla epic-bilder

In [7]:
results = []

for i, path in enumerate(files):
    try:
        img = Image.open(path).convert('RGB').resize(IMG_SIZE)
    except Exception:
        continue
    arr = np.expand_dims(np.array(img), axis=0).astype('float32')
    preds = model.predict(arr, verbose=0)[0]
    p_epic, p_medium, p_thin = float(preds[0]), float(preds[1]), float(preds[2])
    results.append((path, p_epic, p_medium, p_thin))

    if (i + 1) % 500 == 0:
        print(f'{i + 1}/{len(files)} klara...')

# Sortera efter p_medium fallande — högst p_medium (oavsett att epic fortfarande
# vinner) är de mest misstänkta kandidaterna.
results.sort(key=lambda r: r[2], reverse=True)
print('Klart! Topp 5 misstänkta:')
for path, p_epic, p_medium, p_thin in results[:5]:
    print(f'{os.path.basename(path)}: epic={p_epic:.3f} medium={p_medium:.3f} thin={p_thin:.3f}')

500/1437 klara...
1000/1437 klara...
Klart! Topp 5 misstänkta:
11475.JPG: epic=0.027 medium=0.973 thin=0.000
0.000_128694.jpg: epic=0.029 medium=0.971 thin=0.000
11392.JPG: epic=0.036 medium=0.964 thin=0.000
pinterest_54676582962291087.jpg: epic=0.061 medium=0.939 thin=0.000
pinterest_7177680652585557.jpg: epic=0.126 medium=0.874 thin=0.000


## Visa de mest misstänkta kandidaterna
Höj `N` för att se fler. Granska visuellt — ser de mer ut som respektabla/medium-mustascher än episka?

In [ ]:
import matplotlib.pyplot as plt

N = 40
top_candidates = results[:N]

cols = 5
rows = (N + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
axes = np.array(axes).flatten()

for ax, (path, p_epic, p_medium, p_thin) in zip(axes, top_candidates):
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(f'{os.path.basename(path)}\nep={p_epic:.2f} med={p_medium:.2f}', fontsize=8)
    ax.axis('off')

for ax in axes[len(top_candidates):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## Flytta topp 100 till en granskningsmapp
Inte direkt till `medium` — till en granskningsmapp du snabbt sorterar i Finder/Preview efteråt (samma mönster som notebook 16).

In [8]:
import shutil

REVIEW_DIR = 'data/epic_hidden_medium_review'
os.makedirs(REVIEW_DIR, exist_ok=True)

TOP_N = 200
to_move = results[:TOP_N]

moved = 0
for path, p_epic, p_medium, p_thin in to_move:
    fname = os.path.basename(path)
    if os.path.exists(path):
        shutil.move(path, os.path.join(REVIEW_DIR, fname))
        moved += 1

print(f'{moved} bilder (topp {TOP_N} efter p_medium) flyttade till {REVIEW_DIR}.')
print('Gå igenom mappen manuellt — flytta de som genuint är medium till data/epic_dataset/medium,')
print('och flytta resten tillbaka till data/epic_dataset/epic.')

200 bilder (topp 200 efter p_medium) flyttade till data/epic_hidden_medium_review.
Gå igenom mappen manuellt — flytta de som genuint är medium till data/epic_dataset/medium,
och flytta resten tillbaka till data/epic_dataset/epic.
